# KAS-PPI: thực nghiệm mở rộng (08/10/2026)

Notebook này chạy các thực nghiệm còn mở sau buổi báo cáo 07/10. Mọi kết quả được ghi vào thư mục trên Google Drive, nên khi Colab ngắt kết nối thì chạy lại từ đầu: các bước đã xong sẽ được bỏ qua.

| Mã | Thực nghiệm | Câu hỏi | Thời gian ước tính (4 CPU) |
|---|---|---|---|
| E5 | Bảng 9 của KGsim2vec (luật DT6 hay gặp) | Tái hiện phần giải thích của KGsim2vec bằng số | 2 phút |
| E4 | Shapley chính xác so với 100 hoán vị | Ước lượng hoán vị của KAS (k > 8) có đủ chính xác không | 15 phút |
| E1 | So KAS với **bộ giải thích gốc của SEEK** (necessary/sufficient, ánh xạ DCA → aspect) | Câu hỏi hội đồng: "sao chưa so với chính bộ giải thích của SEEK?" | 20 phút (+ giờ nếu phải chạy lại records) |
| E2 | ROAR trên toàn bộ 47,142 cặp cho **view SEEK** (KAS, LOO, SEEK gốc, ngẫu nhiên) | Bổ sung ROAR đầy đủ cho view SEEK | 3–4 giờ |
| E3 | ROAR **hai chiều** MoRF/LeRF + kiểm tra rò rỉ mask, view KGsim2vec | Giảm phụ thuộc vào rò rỉ mask của ROAR một chiều | 1.5 giờ (+2–3 giờ nếu chưa có `all_kg_rf_f*.json`) |

**Cách chạy qua đêm:** Runtime → Change runtime type → CPU, **High-RAM**. Chạy mục 0 (cho phép truy cập Drive), rồi Runtime → Run all.
**Nếu đã có kết quả cũ:** chép thư mục `records/` (các file `rec_seek_*.json`) và các file `all_kg_rf_f*.json` vào thư mục làm việc trên Drive trước khi chạy để bỏ qua bước tính lại.


## 0. Thư mục làm việc trên Drive và cài đặt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/KAS_PPI_mo_rong'
os.makedirs(WORK, exist_ok=True); os.chdir(WORK); print('Thư mục làm việc:', os.getcwd(), '| CPU:', os.cpu_count())

In [ ]:
import os
if not os.path.exists('ExplainablePPI'):
    !git clone -q --depth 1 https://github.com/liseda-lab/ExplainablePPI.git
if not os.path.exists('seek'):
    !git clone -q --depth 1 https://github.com/liseda-lab/seek.git
if not os.path.exists('godata/goa_human.gaf'):
    !mkdir -p godata && unzip -o -q ExplainablePPI/Experimental_Results/Data/go-basic.zip -d godata && unzip -o -q ExplainablePPI/Experimental_Results/Data/goa_human.zip -d godata
if not os.path.exists('seek/PPI/Embeddings/Emb_classes_maxdepth4_nwalks100.txt'):
    !cd seek/PPI && unzip -o -q go.zip && unzip -o -q go_annotations.zip && unzip -o -q Embeddings/Emb_classes_maxdepth4_nwalks100.zip -d Embeddings && unzip -o -q Embeddings/Emb_pair_maxdepth4_nwalks100_Avg_disjointcommonancestor.zip -d Embeddings
!pip -q install shap xgboost lime pyreadr
if os.path.isdir('records'):
    !cp -n records/rec_*.json . 2>/dev/null; true

## Mã nguồn
Mỗi ô ghi một file. Năm file đầu giống notebook `KAS_PPI_colab.ipynb`; các file sau là mới (08/10).

In [ ]:
%%writefile kgsim.py
"""Python re-implementation of KGsim2vec features (Resnik-max, IC Seco computed inside each aspect subgraph)."""
import re, math, collections, pickle, os
OBO="http://purl.obolibrary.org/obo/GO_"
def parse_go(owl):
    parents=collections.defaultdict(set); label={}; cur=None
    pat_cls=re.compile(r'<owl:Class rdf:about="http://purl.obolibrary.org/obo/(GO_\d+)">')
    pat_sub=re.compile(r'<rdfs:subClassOf rdf:resource="http://purl.obolibrary.org/obo/(GO_\d+)"/>')
    pat_lab=re.compile(r'<rdfs:label[^>]*>([^<]*)</rdfs:label>')
    depth=0
    for line in open(owl,encoding="utf-8"):
        m=pat_cls.search(line)
        if m: cur=m.group(1); continue
        if cur is None: continue
        if "</owl:Class>" in line and line.strip()=="</owl:Class>": cur=None; continue
        m=pat_sub.search(line)
        if m: parents[cur].add(m.group(1)); continue
        m=pat_lab.search(line)
        if m and cur not in label: label[cur]=m.group(1)
    return parents,label
def load_gaf(gaf):
    ann=collections.defaultdict(set)
    for l in open(gaf):
        if l.startswith("!"): continue
        f=l.split("\t")
        if len(f)<5 or "NOT" in f[3]: continue
        ann[f[1]].add("GO_"+f[4].split(":")[1])
    return ann
class KG:
    def __init__(s,owl,gaf,aspects):
        s.parents,s.label=parse_go(owl)
        s.children=collections.defaultdict(set)
        for c,ps in s.parents.items():
            for p in ps: s.children[p].add(c)
        s.ann=load_gaf(gaf)
        s.aspects=aspects
        s._anc={}
        s.sub=[]; s.ic=[]
        for a in aspects:
            members=s.desc(a)|{a}
            nprot=sum(1 for p,ts in s.ann.items() if ts & members)
            N=len(members)+nprot   # SML counts annotated instances as vertices of the reduced graph
            ic={}
            for c in members:
                d=len(s.desc(c))
                ic[c]=1-math.log(d+1)/math.log(N) if N>1 else 1.0
            s.sub.append(members); s.ic.append(ic)
    def desc(s,c,_cache={}):
        if c in _cache: return _cache[c]
        out=set(); st=[c]
        while st:
            x=st.pop()
            for y in s.children.get(x,()):
                if y not in out: out.add(y); st.append(y)
        _cache[c]=out; return out
    def anc(s,c):
        if c in s._anc: return s._anc[c]
        out={c}; st=[c]
        while st:
            x=st.pop()
            for y in s.parents.get(x,()):
                if y not in out: out.add(y); st.append(y)
        s._anc[c]=out; return out
    def closure(s,terms):
        out=set()
        for t in terms: out|=s.anc(t)
        return out
    def features(s,t1,t2):
        """t1,t2: sets of direct annotation terms."""
        A1=s.closure(t1); A2=s.closure(t2)
        common=A1&A2
        v=[]
        for i,a in enumerate(s.aspects):
            m=s.sub[i]
            # protein needs >=1 direct annotation in aspect
            if not (t1 & m) or not (t2 & m): v.append(0.0); continue
            cm=common & m
            v.append(max((s.ic[i][c] for c in cm),default=0.0))
        return v


In [ ]:
%%writefile seekrep.py
"""Python re-implementation of SEEK pair representations (Sousa et al., KR 2023):
pair vector = average RDF2Vec embedding of the disjoint common ancestors (DCA) of the two proteins' GO annotations.
Class embeddings are the authors' precomputed ones (fixed); only the DCA set changes when annotations are removed."""
import re, json, collections, numpy as np
from kgsim import parse_go
class SEEK:
    def __init__(s, owl, gaf, emb_file):
        s.parents, s.label = parse_go(owl)
        s.children = collections.defaultdict(set)
        for c, ps in s.parents.items():
            for p in ps: s.children[p].add(c)
        # SEEK's build_kg reads every GAF line (qualifier column ignored), protein id = column 2
        s.ann = collections.defaultdict(set)
        with open(gaf) as f:
            f.readline()
            for l in f:
                if l.startswith("!"): continue
                c = l.split("\t")
                s.ann[c[1]].add("GO_" + c[4].split(":")[1])
        txt = open(emb_file).read().replace("'", '"')
        raw = json.loads(re.sub(r'"http://purl.obolibrary.org/obo/', '"', txt))
        s.emb = {k: np.asarray(v, dtype=np.float64) for k, v in raw.items()}
        s.dim = len(next(iter(s.emb.values())))
        s._anc = {}
    def anc(s, c):
        if c in s._anc: return s._anc[c]
        out = {c}; st = [c]
        while st:
            x = st.pop()
            for y in s.parents.get(x, ()):
                if y not in out: out.add(y); st.append(y)
        s._anc[c] = frozenset(out); return s._anc[c]
    def desc(s, c, _cache={}):
        if c in _cache: return _cache[c]
        out = set(); st = [c]
        while st:
            x = st.pop()
            for y in s.children.get(x, ()):
                if y not in out: out.add(y); st.append(y)
        _cache[c] = out; return out
    def dca(s, t1, t2):
        A1 = set().union(*[s.anc(t) for t in t1]) if t1 else set()
        A2 = set().union(*[s.anc(t) for t in t2]) if t2 else set()
        common = A1 & A2
        strict = set()
        for d in common: strict |= (s.anc(d) - {d})
        return [c for c in common - strict if c in s.emb]
    def rep(s, t1, t2):
        d = s.dca(t1, t2)
        if not d: return np.zeros(s.dim)
        return np.mean([s.emb[c] for c in d], 0)


In [ ]:
%%writefile build_feats.py
import sys,time,numpy as np,pickle
sys.path.insert(0,".")
from kgsim import KG
R="ExplainablePPI/Experimental_Results/"
aspects=[l.split("\t")[0].split("/")[-1] for l in open(R+"Results/kgsim2vec_sas.txt") if l.strip()]
kg=KG("godata/go-basic.owl","godata/goa_human.gaf",aspects)
rows=[l.rstrip("\n").split("\t") for l in open(R+"Results/ss_resnik_max_ICseco.txt")]
lab={}
for l in open(R+"Data/v11(score950).txt"):
    f=l.split(); lab[(f[0],f[1])]=int(f[2])
keys=[(r[0].replace("http://",""),r[1].replace("http://","")) for r in rows]
Xref=np.array([[float(x) for x in r[2:]] for r in rows])
t=time.time()
Xpy=np.array([kg.features(kg.ann[a],kg.ann[b]) for a,b in keys])
print("built",time.time()-t)
y=np.array([lab[k] for k in keys])
np.savez("feats.npz",Xref=Xref,Xpy=Xpy,y=y,keys=np.array(keys),aspects=np.array(aspects))
d=np.abs(Xref-Xpy)
print("exact pairs",(d.max(1)<1e-6).mean(),"cells within 1e-6",(d<1e-6).mean(),"within 0.05",(d<0.05).mean())
print("zero-pattern agreement",((Xref>0)==(Xpy>0)).mean())
cors=[np.corrcoef(Xref[:,j],Xpy[:,j])[0,1] for j in range(52) if Xref[:,j].std()>0 and Xpy[:,j].std()>0]
print("per-aspect pearson median %.4f min %.4f"%(np.median(cors),np.min(cors)))


In [ ]:
%%writefile build_seek_feats.py
import numpy as np, time
from seekrep import SEEK
S="seek/PPI/"
sk=SEEK(S+"go.owl",S+"go_annotations.gaf",S+"Embeddings/Emb_classes_maxdepth4_nwalks100.txt")
K=np.load("feats_kg.npz"); aspects=list(K["aspects"]); keys=[tuple(k) for k in K["keys"]]
roots=["GO_0008150","GO_0003674","GO_0005575"]
ch=set().union(*[sk.children[r] for r in roots])
print("aspects in SEEK GO:",len(ch),"overlap with KG aspects",len(ch&set(aspects)),"missing",set(aspects)-ch)
cx=sk.desc("GO_0032991")|{"GO_0032991"}
Xs=np.zeros((len(keys),sk.dim)); Xs51=np.zeros_like(Xs); t=time.time()
for i,(a,b) in enumerate(keys):
    t1,t2=sk.ann[a],sk.ann[b]
    Xs[i]=sk.rep(t1,t2); Xs51[i]=sk.rep(t1-cx,t2-cx)
print("built",time.time()-t, "rows changed by complex removal",(np.abs(Xs-Xs51).max(1)>0).mean())
np.savez("feats_seek.npz",Xseek=Xs,Xseek51=Xs51)


In [ ]:
%%writefile explain_v2.py
"""Direction D, full basic experiment: knowledge-space aspect Shapley (D) on three pipelines.
usage: python3 explain_v2.py PIPE MODEL FOLD N    PIPE in {kg,seek,hyb}, MODEL in {rf,xgb}
Players = GO aspects (51 direct children of the 3 GO roots, 'protein-containing complex' removed) in which
both proteins have >=1 direct annotation. Removing a player = deleting both proteins' annotations under that
aspect and recomputing the features through the pipeline (KGsim2vec: Resnik-max/IC-Seco; SEEK: DCA + mean
RDF2Vec embedding with the authors' fixed class embeddings)."""
import sys, time, json, itertools, math, random
import numpy as np
sys.path.insert(0,"."); sys.path.insert(0,"ExplainablePPI")
from kgsim import KG
from seekrep import SEEK
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from scipy.stats import spearmanr
import shap
from lime import lime_tabular

PIPE,MODEL,FOLD,N_PAIRS=sys.argv[1],sys.argv[2],int(sys.argv[3]),int(sys.argv[4])
M_PERM=100; EXACT_MAX=8; TOPK=5
rng=np.random.default_rng(FOLD); random.seed(FOLD)

K=np.load("feats_kg.npz"); y=K["y"]; keys=[tuple(k) for k in K["keys"]]; aspects=list(K["aspects"])
ci=aspects.index("GO_0032991"); keep=[j for j in range(52) if j!=ci]; A=[aspects[j] for j in keep]
SF=np.load("feats_seek.npz")
X={"kg":K["Xpy"][:,keep],"seek":SF["Xseek51"],"hyb":np.hstack([K["Xpy"][:,keep],SF["Xseek51"]])}[PIPE]
kg=KG("godata/go-basic.owl","godata/goa_human.gaf",aspects); sub_kg=[kg.sub[j] for j in keep]
S_="seek/PPI/"
sk=SEEK(S_+"go.owl",S_+"go_annotations.gaf",S_+"Embeddings/Emb_classes_maxdepth4_nwalks100.txt")
CX=sk.desc("GO_0032991")|{"GO_0032991"}
sub_sk=[sk.desc(a)|{a} for a in A]

P="ExplainablePPI/Experimental_Results/Data/StratifiedPartitions/Indexes__crossvalidationTest__Run"
test=[int(l) for l in open(P+("1(complete)" if FOLD==1 else str(FOLD))+".txt") if l.strip()]
m=np.zeros(len(y),bool); m[test]=True
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
clf=(RandomForestClassifier(100,n_jobs=1,random_state=0) if MODEL=="rf"
     else XGBClassifier(n_estimators=300,n_jobs=1,random_state=0,eval_metric="logloss") if MODEL=="xgb"
     else make_pipeline(StandardScaler(),MLPClassifier((100,),max_iter=300,early_stopping=True,random_state=0))).fit(X[~m],y[~m])
def proba(Z): return clf.predict_proba(np.atleast_2d(Z))[:,1].astype(np.float64)

# state = (kg annotations p1, kg annotations p2, seek annotations p1, seek annotations p2)
def feats(st):
    k1,k2,s1,s2=st; parts=[]
    if PIPE in("kg","hyb"):
        v=kg.features(k1,k2); parts.append([v[j] for j in keep])
    if PIPE in("seek","hyb"): parts.append(list(sk.rep(s1,s2)))
    return np.concatenate(parts)
def players_of(st):
    k1,k2,s1,s2=st; out=[]
    for i in range(len(A)):
        a=(PIPE!="seek") and bool(k1&sub_kg[i]) and bool(k2&sub_kg[i])
        b=(PIPE!="kg") and bool(s1&sub_sk[i]) and bool(s2&sub_sk[i])
        if a or b: out.append(i)
    return out
def remove(st,S):
    if not S: return st
    rk=set().union(*[sub_kg[i] for i in S]); rs=set().union(*[sub_sk[i] for i in S])
    k1,k2,s1,s2=st; return (k1-rk,k2-rk,s1-rs,s2-rs)

def shapley(value_batch, players, seed):
    k=len(players); phi=np.zeros(len(A))
    if k==0: return phi
    if k<=EXACT_MAX:
        subsets=[frozenset(c) for r in range(k+1) for c in itertools.combinations(players,r)]
        vals=dict(zip(subsets,value_batch(subsets)))
        for i in players:
            others=[p for p in players if p!=i]
            for r in range(k):
                w=math.factorial(r)*math.factorial(k-r-1)/math.factorial(k)
                for c in itertools.combinations(others,r):
                    S=frozenset(c); phi[i]+=w*(vals[S]-vals[S|{i}])
        return phi
    r=random.Random(seed); perms=[r.sample(players,k) for _ in range(M_PERM)]
    subsets=set()
    for p in perms:
        for t in range(k+1): subsets.add(frozenset(p[:t]))
    subsets=list(subsets); vals=dict(zip(subsets,value_batch(subsets)))
    for p in perms:
        for t,i in enumerate(p): phi[i]+=vals[frozenset(p[:t])]-vals[frozenset(p[:t+1])]
    return phi/M_PERM

def explain_D(st,seed=0):
    pl=players_of(st)
    return shapley(lambda subs: proba(np.array([feats(remove(st,S)) for S in subs])),pl,seed)
def explain_LOO(st,seed=0):   # leave-one-aspect-out occlusion in knowledge space (aspect-level analogue of SEEK 'necessary')
    pl=players_of(st); phi=np.zeros(len(A))
    if not pl: return phi
    v=proba(np.array([feats(st)]+[feats(remove(st,[i])) for i in pl]))
    for j,i in enumerate(pl): phi[i]=v[0]-v[j+1]
    return phi
def explain_FS0(st,seed=0):   # KGsim2vec only: same players and sampler, feature set to 0 instead of recomputed
    pl=players_of(st); x=feats(st)
    def vb(subs):
        Z=np.repeat(x[None],len(subs),0)
        for r,S in enumerate(subs): Z[r,list(S)]=0
        return proba(Z)
    return shapley(vb,pl,seed)
bg=X[~m][rng.choice((~m).sum(),100,replace=False)]
tree=shap.TreeExplainer(clf,data=bg,feature_perturbation="interventional") if MODEL!="mlp" else None
def explain_TS(st,seed=0):    # kg: per aspect feature; hyb: only the 51 named KGsim2vec columns map to aspects
    x=feats(st); sv=tree.shap_values(x[None],check_additivity=False)
    sv=sv[1] if isinstance(sv,list) else sv
    sv=np.array(sv); sv=sv[0,:,1] if sv.ndim==3 else sv[0]
    return np.asarray(sv[:len(A)],dtype=np.float64)
if PIPE=="kg":
    lime_exp=lime_tabular.LimeTabularExplainer(X[~m][:2000],feature_names=A,class_names=["0","1"],discretize_continuous=True,random_state=0)
def explain_LIME(st,seed=0):
    x=feats(st); lime_exp.random_state=np.random.RandomState(seed)
    e=lime_exp.explain_instance(x,lambda Z: clf.predict_proba(Z).astype(np.float64),num_features=10,num_samples=2000,labels=(1,))
    phi=np.zeros(len(A))
    for j,w in e.as_map()[1]: phi[j]=w
    return phi
METHODS={"D":explain_D,"LOO":explain_LOO}
if PIPE=="kg": METHODS.update({"FS0":explain_FS0,"TreeSHAP":explain_TS,"LIME":explain_LIME})
if PIPE=="hyb": METHODS.update({"TreeSHAP_KGpart":explain_TS})
if MODEL=="mlp": METHODS={k:v for k,v in METHODS.items() if not k.startswith("TreeSHAP")}
DETERMINISTIC={"LOO","TreeSHAP","TreeSHAP_KGpart"}

def ranking(phi,pred):
    s=phi if pred==1 else -phi
    return [int(j) for j in np.argsort(-s) if s[j]>0]
def deletion_ks(st,order,pred):
    Z=np.array([feats(remove(st,order[:k])) for k in range(1,TOPK+1)]); p=proba(Z)
    return list(p if pred==1 else 1-p)
def insertion_ks(st,order,pred,pl):  # start with every player removed, add back the top-k
    Z=np.array([feats(remove(st,[i for i in pl if i not in order[:k]])) for k in range(0,TOPK+1)]); p=proba(Z)
    return list(p if pred==1 else 1-p)
def eff_size(phi):
    a=np.sort(np.abs(phi))[::-1]; tot=a.sum()
    return int(np.searchsorted(np.cumsum(a)/tot,0.8)+1) if tot>0 else 0
def jacc(a,b,k=3):
    A_,B_=set(a[:k]),set(b[:k]); return len(A_&B_)/len(A_|B_) if (A_|B_) else 1.0

pos=[i for i in test if y[i]==1]; neg=[i for i in test if y[i]==0]
idx=list(rng.choice(pos,N_PAIRS//2,replace=False))+list(rng.choice(neg,N_PAIRS//2,replace=False))
rec=[]; t0=time.time()
for n,i in enumerate(idx):
    p1,p2=keys[i]
    st=(set(kg.ann[p1]),set(kg.ann[p2]),set(sk.ann[p1])-CX,set(sk.ann[p2])-CX)
    p=proba(feats(st))[0]; pred=int(p>0.5); conf=p if pred else 1-p
    pl=players_of(st)
    r=dict(i=int(i),pair=f"{p1}-{p2}",y=int(y[i]),pred=pred,conf=float(conf),k=len(pl))
    # continuity: drop ONE random direct annotation (same GO term in both annotation sets) lying in a player aspect
    cand=sorted({(w,t) for w in (0,1) for t in (st[w]|st[w+2]) if any(t in sub_kg[a] or t in sub_sk[a] for a in pl)})
    pert=None
    if cand:
        w,t=cand[rng.integers(len(cand))]; q=list(st); q[w]=q[w]-{t}; q[w+2]=q[w+2]-{t}; q=tuple(q)
        if int(proba(feats(q))[0]>0.5)==pred: pert=q
    ro=[int(v) for v in rng.permutation(pl)] if pl else []
    r["del_random"]=deletion_ks(st,ro,pred) if ro else None
    r["ins_random"]=insertion_ks(st,ro,pred,pl) if ro else None
    for name,f in METHODS.items():
        ts_=time.time(); phi=f(st,0); r[f"time_{name}"]=time.time()-ts_
        order=ranking(phi,pred)
        r[f"cov_{name}"]=int(len(order)>0); r[f"size_{name}"]=eff_size(phi)
        r[f"del_{name}"]=deletion_ks(st,order,pred) if order else None
        r[f"ins_{name}"]=insertion_ks(st,order,pred,pl) if order else None
        r[f"top3_{name}"]=[A[j] for j in order[:3]]
        r[f"phi_{name}"]={A[j]:round(float(phi[j]),5) for j in np.nonzero(phi)[0]}
        if name in DETERMINISTIC or (name=="D" and len(pl)<=EXACT_MAX):
            r[f"cons_jacc_{name}"]=1.0; r[f"cons_rho_{name}"]=1.0
        else:
            phi2=f(st,1); o2=ranking(phi2,pred); r[f"cons_jacc_{name}"]=jacc(order,o2)
            r[f"cons_rho_{name}"]=float(spearmanr(phi,phi2).correlation) if np.std(phi)>0 and np.std(phi2)>0 else None
        r[f"cont_{name}"]=jacc(order,ranking(f(pert,0),pred)) if pert is not None else None
    rec.append(r)
    if n%25==0: print(PIPE,MODEL,FOLD,n,round(time.time()-t0,1),"s k=",len(pl),flush=True)
json.dump(dict(pipe=PIPE,model=MODEL,fold=FOLD,aspects=A,labels={a:kg.label.get(a,a) for a in A},records=rec),open(f"rec_{PIPE}_{MODEL}_f{FOLD}_{N_PAIRS}.json","w"))
print("done",time.time()-t0)


In [ ]:
%%writefile explain_all.py
"""Explain EVERY test pair of one fold (KGsim2vec view) with KAS, LOO, FS0, TreeSHAP, LIME, for full-data ROAR.
usage: python3 explain_all.py MODEL FOLD [PART NPARTS]   -> all_kg_{MODEL}_f{FOLD}[_p{PART}].json
(each pair explained by the fold model that did not train on it; PART/NPARTS split a fold across CPUs)"""
import sys, json, time
import numpy as np
MODEL, FOLD = sys.argv[1], int(sys.argv[2])
PART, NPARTS = (int(sys.argv[3]), int(sys.argv[4])) if len(sys.argv) > 4 else (0, 1)
sys.argv = ["x", "kg", MODEL, str(FOLD), "1"]
exec(open("explain_v2.py").read().split("pos=[i for i in test")[0])   # setup: data, model, feats, explainers
out = []; t0 = time.time()
for n, i in enumerate(test[PART::NPARTS]):
    p1, p2 = keys[i]
    st = (set(kg.ann[p1]), set(kg.ann[p2]), set(), set())
    p = proba(feats(st))[0]; r = dict(i=int(i), y=int(y[i]), pred=int(p > 0.5))
    for name, f in METHODS.items():
        phi = f(st, 0); r[f"phi_{name}"] = {A[j]: round(float(phi[j]), 5) for j in np.nonzero(phi)[0]}
    out.append(r)
    if n % 500 == 0: print(FOLD, n, round(time.time() - t0), "s", flush=True)
json.dump(dict(model=MODEL, fold=FOLD, aspects=A, records=out), open(f"all_kg_{MODEL}_f{FOLD}" + (f"_p{PART}" if NPARTS > 1 else "") + ".json", "w"))
print("done", FOLD, time.time() - t0)


In [ ]:
%%writefile get_msigdb.py
"""Extract Reactome and KEGG gene sets (gene symbols) from MSigDB 7.5.1 as bundled in the conda-forge package r-msigdbr 7.5.1
(R lazy-load database, LZMA-compressed objects). Writes REACTOME_msigdb751.gmt and KEGG_msigdb751.gmt. Needs pyreadr."""
import os, io, lzma, struct, tarfile, urllib.request, tempfile, pyreadr
URL = "https://conda.anaconda.org/conda-forge/noarch/r-msigdbr-7.5.1-r42hc72bb7e_1.tar.bz2"
tmp = tempfile.mkdtemp()
tb = os.path.join(tmp, "m.tar.bz2"); urllib.request.urlretrieve(URL, tb)
with tarfile.open(tb) as t:
    b = t.extractfile("lib/R/library/msigdbr/R/sysdata.rdb").read()
objs = []; off = 0
while off < len(b):
    dec = lzma.LZMADecompressor(format=lzma.FORMAT_RAW, filters=[{"id": lzma.FILTER_LZMA2, "preset": 9 | lzma.PRESET_EXTREME}])
    d = dec.decompress(b[off + 5:]); used = len(b) - off - 5 - len(dec.unused_data)
    p = os.path.join(tmp, f"o{len(objs)}.rds"); open(p, "wb").write(d); objs.append(pyreadr.read_r(p)[None]); off += 5 + used
genes, gsg, gs = objs  # msigdbr_genes, msigdbr_geneset_genes, msigdbr_genesets
for sub in ["CP:REACTOME", "CP:KEGG"]:
    s = gs[gs.gs_subcat == sub]; m = gsg[gsg.gs_id.isin(s.gs_id)].merge(genes, on="gene_id"); name = s.set_index("gs_id")["gs_name"]
    with open(sub.split(":")[1] + "_msigdb751.gmt", "w") as f:
        for gid, grp in m.groupby("gs_id"):
            f.write(name[gid] + "\t" + gid + "\t" + "\t".join(sorted(set(grp.human_gene_symbol))) + "\n")
    print(sub, len(s), "gene sets")


In [ ]:
%%writefile seeknat.py
"""SEEK's own explainer (Sousa et al. 2023, generate_seek_explanation.py), made graded and mapped to the 51 fixed aspects.
SEEK explains in DCA space: for each disjoint common ancestor c of the pair it asks the model
  'without c'  (average embedding of the other DCAs; zero vector if none)  -> c is NECESSARY if the predicted class flips
  'only c'     (embedding of c alone)                                        -> c is SUFFICIENT if the predicted class stays.
Graded versions used here (toward interaction, i.e. class-1 probability):
  SEEKN(c) = p(all DCAs) - p(without c)      SEEKS(c) = p(only c) - 0.5
Aspect score: SEEKN(a) = sum of SEEKN(c) over the DCAs c lying in aspect a's subtree;
              SEEKS(a) = SEEKS(c*) for the DCA c* in a that most supports the predicted class.
Each DCA of a pair lies only in player aspects (both proteins have descendants of c), so the mapping never leaves the players.
DCAs outside every aspect (e.g. the BP/MF/CC roots) are counted but cannot be attributed."""
import numpy as np

def dca_vec(sk, D):
    return np.mean([sk.emb[c] for c in D], 0) if D else np.zeros(sk.dim)

def native(st, sk, sub_sk, proba, nA):
    s1, s2 = st[2], st[3]
    D = sk.dca(s1, s2)
    Z = [dca_vec(sk, D)] + [dca_vec(sk, [d for d in D if d != c]) for c in D] + [sk.emb[c] for c in D]
    p = proba(np.array(Z)); p0 = p[0]; m = len(D)
    pw, po = p[1:1 + m], p[1 + m:]
    pred = int(p0 > 0.5); sgn = 1 if pred else -1
    nec = [c for c, q in zip(D, pw) if int(q > 0.5) != pred]
    suf = [c for c, q in zip(D, po) if int(q > 0.5) == pred]
    asp_of = {c: [a for a in range(nA) if c in sub_sk[a]] for c in D}
    phiN = np.zeros(nA); phiS = np.zeros(nA); best = np.full(nA, -np.inf)
    for c, q_w, q_o in zip(D, pw, po):
        for a in asp_of[c]:
            phiN[a] += p0 - q_w
            v = q_o - 0.5
            if sgn * v > best[a]: best[a] = sgn * v; phiS[a] = v
    stats = dict(n_dca=m, n_dca_noasp=sum(1 for c in D if not asp_of[c]), n_nec=len(nec), n_suf=len(suf),
                 nec_asp=sorted({a for c in nec for a in asp_of[c]}), suf_asp=sorted({a for c in suf for a in asp_of[c]}))
    return phiN, phiS, stats, D, asp_of

def dca_curves(D, asp_of, order, pl, pred, sk, proba, topk=5):
    """Deletion / insertion in SEEK's own (DCA) space: drop the DCAs that lie in the removed aspects and average the
    rest, without recomputing annotations. Returns predicted-class probabilities (deletion k=1..topk, insertion k=0..topk)."""
    def keep_without(S):
        S = set(S); return [c for c in D if not (set(asp_of[c]) & S)]
    Zd = [dca_vec(sk, keep_without(order[:k])) for k in range(1, topk + 1)]
    Zi = [dca_vec(sk, keep_without([a for a in pl if a not in order[:k]])) for k in range(0, topk + 1)]
    p = proba(np.array(Zd + Zi)); p = p if pred == 1 else 1 - p
    return list(p[:topk]), list(p[topk:])


In [ ]:
%%writefile seek_native.py
"""E1: SEEK's own explainer vs KAS on the SEEK view, same 2,200 pairs per model as the main runs (rec_seek_{MODEL}_f*.json).
usage: python3 seek_native.py MODEL FOLD      -> rec_seekx_{MODEL}_f{FOLD}.json
Two removal operators: 'ks' = delete annotations + recompute (KAS's value function, same code as explain_v2),
'dca' = drop DCA vectors of the removed aspects, no recompute (SEEK's own operator)."""
import sys, json, glob, time
import numpy as np
MODEL, FOLD = sys.argv[1], int(sys.argv[2])
sys.argv = ["x", "seek", MODEL, str(FOLD), "1"]
exec(open("explain_v2.py").read().split("pos=[i for i in test")[0])
from seeknat import native, dca_curves
recs = []
for fn in glob.glob(f"rec_seek_{MODEL}_f{FOLD}_*.json"): recs += json.load(open(fn))["records"]
out = []; t0 = time.time()
for n, r in enumerate(recs):
    i = r["i"]; p1, p2 = keys[i]
    st = (set(kg.ann[p1]), set(kg.ann[p2]), set(sk.ann[p1]) - CX, set(sk.ann[p2]) - CX)
    pred = r["pred"]; pl = players_of(st)
    ts = time.time(); phiN, phiS, stats, D, asp_of = native(st, sk, sub_sk, proba, len(A)); tn = time.time() - ts
    phis = {"D": np.array([r["phi_D"].get(a, 0.0) for a in A]), "LOO": np.array([r["phi_LOO"].get(a, 0.0) for a in A]),
            "SEEKN": phiN, "SEEKS": phiS}
    o = dict(i=i, pair=r["pair"], y=r["y"], pred=pred, k=len(pl), time_SEEK=tn, **stats)
    rr = np.random.default_rng(i); orders = {m: ranking(f, pred) for m, f in phis.items()}
    orders["random"] = [int(v) for v in rr.permutation(pl)]
    for m, f in phis.items():
        o[f"phi_{m}"] = {A[j]: round(float(f[j]), 5) for j in np.nonzero(f)[0]}
        o[f"size_{m}"] = eff_size(f); o[f"cov_{m}"] = int(len(orders[m]) > 0)
    for m, od in orders.items():
        if not od: continue
        if m in ("D", "LOO"): o[f"del_ks_{m}"], o[f"ins_ks_{m}"] = r[f"del_{m}"], r[f"ins_{m}"]
        else: o[f"del_ks_{m}"], o[f"ins_ks_{m}"] = deletion_ks(st, od, pred), insertion_ks(st, od, pred, pl)
        o[f"del_dca_{m}"], o[f"ins_dca_{m}"] = dca_curves(D, asp_of, od, pl, pred, sk, proba)
    o["top1_D"] = A[orders["D"][0]] if orders["D"] else None
    for m in ("LOO", "SEEKN", "SEEKS"):
        a, b = orders["D"][:3], orders[m][:3]
        o[f"jac3_D_{m}"] = len(set(a) & set(b)) / len(set(a) | set(b)) if (a or b) else 1.0
        o[f"top1same_D_{m}"] = int(bool(a) and bool(b) and a[0] == b[0])
    out.append({k: (float(v) if isinstance(v, np.floating) else v) for k, v in o.items()})
    if n % 100 == 0: print(MODEL, FOLD, n, round(time.time() - t0), "s", flush=True)
json.dump(dict(pipe="seekx", model=MODEL, fold=FOLD, aspects=A, records=out), open(f"rec_seekx_{MODEL}_f{FOLD}.json", "w"))
print("done", MODEL, FOLD, round(time.time() - t0))


In [ ]:
%%writefile explain_all_seek.py
"""E2 input: explain EVERY test pair of one fold on the SEEK view with KAS (D), LOO and SEEK's own explainer (SEEKN, SEEKS),
for full-data ROAR on SEEK. usage: python3 explain_all_seek.py MODEL FOLD [PART NPARTS] -> all_seek_{MODEL}_f{FOLD}[_p{PART}].json
(each pair explained by the fold model that did not train on it)."""
import sys, json, time
import numpy as np
MODEL, FOLD = sys.argv[1], int(sys.argv[2])
PART, NPARTS = (int(sys.argv[3]), int(sys.argv[4])) if len(sys.argv) > 4 else (0, 1)
sys.argv = ["x", "seek", MODEL, str(FOLD), "1"]
exec(open("explain_v2.py").read().split("pos=[i for i in test")[0])
from seeknat import native
out = []; t0 = time.time()
for n, i in enumerate(test[PART::NPARTS]):
    p1, p2 = keys[i]
    st = (set(kg.ann[p1]), set(kg.ann[p2]), set(sk.ann[p1]) - CX, set(sk.ann[p2]) - CX)
    p = proba(feats(st))[0]; r = dict(i=int(i), y=int(y[i]), pred=int(p > 0.5))
    phiN, phiS, stats, _, _ = native(st, sk, sub_sk, proba, len(A))
    for name, phi in (("D", METHODS["D"](st, 0)), ("LOO", METHODS["LOO"](st, 0)), ("SEEKN", phiN), ("SEEKS", phiS)):
        r[f"phi_{name}"] = {A[j]: round(float(phi[j]), 5) for j in np.nonzero(phi)[0]}
    r["n_nec"], r["n_suf"], r["n_dca"] = stats["n_nec"], stats["n_suf"], stats["n_dca"]
    out.append(r)
    if n % 500 == 0: print(FOLD, PART, n, round(time.time() - t0), "s", flush=True)
json.dump(dict(model=MODEL, fold=FOLD, aspects=A, records=out),
          open(f"all_seek_{MODEL}_f{FOLD}" + (f"_p{PART}" if NPARTS > 1 else "") + ".json", "w"))
print("done", FOLD, PART, round(time.time() - t0))


In [ ]:
%%writefile roar_sym.py
"""E2/E3: full-data ROAR in BOTH directions (MoRF = remove the k most relevant aspects; LeRF = remove the k least relevant
players), on all 47,142 pairs, with retraining on the repo's 10 partitions, plus the mask-only leakage check for each.
A good ranking makes MoRF hurt a lot AND LeRF hurt little; reporting both (and their gap) is the usual answer to the
class-information leakage that makes single-direction ROAR unreliable (discussed by Rong et al. 2022, ROAD).
usage: python3 roar_sym.py PIPE MODEL   PIPE in {kg, seek}; reads all_{PIPE}_{MODEL}_f*.json -> roar_sym_{PIPE}_{MODEL}.json
Orders use the signed attribution toward the pair's predicted class; ties broken at random (seeded per pair)."""
import sys, glob, json, random, time
import numpy as np
sys.path.insert(0, "."); sys.path.insert(0, "ExplainablePPI")
from kgsim import KG
from seekrep import SEEK
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
from scipy.stats import wilcoxon
PIPE, MODEL = sys.argv[1], sys.argv[2]; KS = [1, 2, 3]
K = np.load("feats_kg.npz"); keys = [tuple(k) for k in K["keys"]]; aspects = list(K["aspects"]); y = K["y"]
ci = aspects.index("GO_0032991"); keep = [j for j in range(52) if j != ci]; A = [aspects[j] for j in keep]
if PIPE == "kg":
    kg = KG("godata/go-basic.owl", "godata/goa_human.gaf", aspects); sub = [kg.sub[j] for j in keep]
    X0 = K["Xpy"][:, keep]; ann = lambda p: kg.ann[p]
    def feats(t1, t2): v = kg.features(t1, t2); return [v[j] for j in keep]
else:
    S_ = "seek/PPI/"
    sk = SEEK(S_ + "go.owl", S_ + "go_annotations.gaf", S_ + "Embeddings/Emb_classes_maxdepth4_nwalks100.txt")
    CX = sk.desc("GO_0032991") | {"GO_0032991"}; sub = [sk.desc(a) | {a} for a in A]
    X0 = np.load("feats_seek.npz")["Xseek51"]; ann = lambda p: set(sk.ann[p]) - CX
    def feats(t1, t2): return sk.rep(t1, t2)
rec = {}
for f in sorted(glob.glob(f"all_{PIPE}_{MODEL}_f*.json")):
    for r in json.load(open(f))["records"]: rec[r["i"]] = r
assert len(rec) == len(y), (len(rec), len(y))
methods = [k[4:] for k in rec[0] if k.startswith("phi_")]
variants = [(m, d) for m in methods for d in ("MoRF", "LeRF")] + [("random", "MoRF")]
Xs = {(m, d, k): np.zeros_like(X0) for m, d in variants for k in KS}
Ms = {(m, d, k): np.zeros((len(y), len(A)), np.int8) for m, d in variants for k in KS}
t0 = time.time(); cache = {}
for i in range(len(y)):
    r = rec[i]; p1, p2 = keys[i]; t1, t2 = ann(p1), ann(p2)
    pl = [a for a in range(len(A)) if t1 & sub[a] and t2 & sub[a]]
    sgn = 1 if r["pred"] == 1 else -1; rr = random.Random(i); tb = {a: rr.random() for a in pl}
    cache.clear()
    for m, d in variants:
        if m == "random": order = random.Random(i + 7).sample(pl, len(pl))
        else:
            phi = r[f"phi_{m}"]; s = {a: sgn * phi.get(A[a], 0.0) for a in pl}
            order = sorted(pl, key=lambda a: (-s[a], tb[a])) if d == "MoRF" else sorted(pl, key=lambda a: (s[a], tb[a]))
        for k in KS:
            S = tuple(sorted(order[:k]))
            if S:
                if S not in cache:
                    U = set().union(*[sub[a] for a in S]); cache[S] = feats(t1 - U, t2 - U)
                Xs[(m, d, k)][i] = cache[S]; Ms[(m, d, k)][i, list(S)] = 1
            else: Xs[(m, d, k)][i] = X0[i]
    if i % 5000 == 0: print(i, round(time.time() - t0), "s", flush=True)
P = "ExplainablePPI/Experimental_Results/Data/StratifiedPartitions/Indexes__crossvalidationTest__Run"
folds = [[int(l) for l in open(P + ("1(complete)" if f == 1 else str(f)) + ".txt") if l.strip()] for f in range(1, 11)]
NJ = int(__import__("os").environ.get("NJOBS", "4"))
def make(): return (RandomForestClassifier(100, n_jobs=NJ, random_state=0) if MODEL == "rf"
                    else XGBClassifier(n_estimators=300, n_jobs=NJ, random_state=0, eval_metric="logloss"))
def cv(X, mk=make):
    f1s = []
    for te in folds:
        m = np.zeros(len(y), bool); m[te] = True
        c = mk().fit(X[~m], y[~m]); f1s.append(f1_score(y[m], c.predict(X[m]), average="weighted"))
    return f1s
out = dict(pipe=PIPE, model=MODEL, n=len(y), base=cv(X0), res={}, mask={})
base = float(np.median(out["base"])); print("base WAF %.4f" % base, flush=True)
for m, d in variants:
    for k in KS:
        key = f"{m}|{d}|{k}"
        out["res"][key] = cv(Xs[(m, d, k)])
        out["mask"][key] = float(np.mean(cv(Ms[(m, d, k)], lambda: RandomForestClassifier(100, n_jobs=NJ, random_state=0))))
        print(key, "WAF %.4f drop %.4f mask-only %.3f" % (np.median(out["res"][key]), base - np.median(out["res"][key]), out["mask"][key]),
              round(time.time() - t0), "s", flush=True)
        json.dump(out, open(f"roar_sym_{PIPE}_{MODEL}.json", "w"))
# per-fold gap = WAF(LeRF) - WAF(MoRF): larger = ranking separates relevant from irrelevant aspects
out["gap"] = {}; out["p"] = {}
for k in KS:
    for m in methods:
        out["gap"][f"{m}|{k}"] = list(np.array(out["res"][f"{m}|LeRF|{k}"]) - np.array(out["res"][f"{m}|MoRF|{k}"]))
    for m in methods:
        if m == "D": continue
        for name, a, b in (("MoRF_less", out["res"][f"D|MoRF|{k}"], out["res"][f"{m}|MoRF|{k}"]),
                           ("LeRF_greater", out["res"][f"D|LeRF|{k}"], out["res"][f"{m}|LeRF|{k}"]),
                           ("gap_greater", out["gap"][f"D|{k}"], out["gap"][f"{m}|{k}"])):
            alt = "less" if name == "MoRF_less" else "greater"
            out["p"][f"D_{name}_{m}_{k}"] = float(wilcoxon(a, b, alternative=alt).pvalue) if np.any(np.array(a) != np.array(b)) else 1.0
json.dump(out, open(f"roar_sym_{PIPE}_{MODEL}.json", "w"))
print("FINISHED")


In [ ]:
%%writefile perm_check.py
"""E4: accuracy of the 100-permutation Shapley estimate used by KAS when k > 8. On pairs with 6 <= k <= 8 (exact is
affordable) compare exact KAS with permutation estimates (M = 25, 50, 100, 200; 3 seeds each), same value function.
usage: python3 perm_check.py PIPE MODEL FOLD N    -> perm_{PIPE}_{MODEL}_f{FOLD}.json"""
import sys, json, time
import numpy as np
from scipy.stats import spearmanr
PIPE_, MODEL_, FOLD_, N_ = sys.argv[1], sys.argv[2], int(sys.argv[3]), int(sys.argv[4])
sys.argv = ["x", PIPE_, MODEL_, str(FOLD_), "1"]
exec(open("explain_v2.py").read().split("pos=[i for i in test")[0])
rs = np.random.default_rng(1000 + FOLD_); cand = list(rs.permutation(test)); res = []; t0 = time.time()
for i in cand:
    if len(res) >= N_: break
    p1, p2 = keys[i]
    st = (set(kg.ann[p1]), set(kg.ann[p2]), set(sk.ann[p1]) - CX, set(sk.ann[p2]) - CX)
    pl = players_of(st)
    if not 6 <= len(pl) <= 8: continue
    cache = {}
    def vb(subs):
        need = [S for S in subs if S not in cache]
        if need:
            for S, v in zip(need, proba(np.array([feats(remove(st, S)) for S in need]))): cache[S] = v
        return [cache[S] for S in subs]
    EXACT_MAX = 8; ex = shapley(vb, pl, 0)
    pred = int(cache[frozenset()] > 0.5); sgn = 1 if pred else -1
    top = lambda f: [int(j) for j in np.argsort(-sgn * f) if j in pl]
    o_ex = top(ex); scale = np.abs(ex[pl]).max() or 1.0
    row = dict(i=int(i), k=len(pl))
    for M in (25, 50, 100, 200):
        for sd in range(3):
            EXACT_MAX = 0; M_PERM = M; est = shapley(vb, pl, 17 * sd + M)
            o = top(est)
            row[f"M{M}_s{sd}"] = dict(rel_mae=float(np.abs(est[pl] - ex[pl]).mean() / scale),
                                      rho=float(spearmanr(est[pl], ex[pl]).correlation) if np.std(ex[pl]) > 0 and np.std(est[pl]) > 0 else 1.0,
                                      top1=int(o[0] == o_ex[0]), jac3=len(set(o[:3]) & set(o_ex[:3])) / len(set(o[:3]) | set(o_ex[:3])),
                                      sum_err=float(abs(est.sum() - ex.sum())))
    EXACT_MAX = 8; M_PERM = 100
    res.append(row)
    if len(res) % 50 == 0: print(len(res), round(time.time() - t0), "s", flush=True)
summ = {}
for M in (25, 50, 100, 200):
    vals = [r[f"M{M}_s{sd}"] for r in res for sd in range(3)]
    summ[M] = {k: float(np.mean([v[k] for v in vals])) for k in ("rel_mae", "rho", "top1", "jac3", "sum_err")}
    print(PIPE_, MODEL_, "M=%d" % M, " ".join("%s %.3f" % kv for kv in summ[M].items()))
json.dump(dict(pipe=PIPE_, model=MODEL_, fold=FOLD_, n=len(res), summary=summ, rows=res), open(f"perm_{PIPE_}_{MODEL_}_f{FOLD_}.json", "w"))


In [ ]:
%%writefile dt6_rules.py
"""E5: KGsim2vec Table 9 (most frequent DT6 rules) reproduction.
Protocol of Sousa et al. 2024 (sec. 4.6 + Prediction/ML.py, processDTs.py): DT with max_depth=6 on the SML similarity
features of 51 aspects, one model per repo partition (10 models); sibling leaves with the same decision are pruned as in
prune_duplicate_leaves; each leaf gives a rule (per feature: max of '>' cut-offs, min of '<=' cut-offs); cut-offs rounded
to one decimal; a rule's frequency = number of models containing it; #pairs = test pairs reaching that rule, summed over
the models that contain it. Rules in >= 5 models are compared with the paper's Table 9.
usage: python3 dt6_rules.py [SEEDS]   (default 0; e.g. 0,1,2,3,4 to check stability) -> dt6_rules.json"""
import sys, json, collections, re
import numpy as np
sys.path.insert(0, ".")
from kgsim import parse_go
from sklearn.tree import DecisionTreeClassifier
SEEDS = [int(s) for s in (sys.argv[1] if len(sys.argv) > 1 else "0").split(",")]
K = np.load("feats_kg.npz"); y = K["y"]; aspects = list(K["aspects"])
ci = aspects.index("GO_0032991"); keep = [j for j in range(52) if j != ci]
_, label = parse_go("godata/go-basic.owl")
names = [label[aspects[j]].replace(" ", "_") for j in keep]
X = K["Xref"][:, keep]
P = "ExplainablePPI/Experimental_Results/Data/StratifiedPartitions/Indexes__crossvalidationTest__Run"
folds = [[int(l) for l in open(P + ("1(complete)" if f == 1 else str(f)) + ".txt") if l.strip()] for f in range(1, 11)]
PAPER = """1|binding <= 0.5 & 0.0 < cellular_process <= 0.6 & biological_regulation <= 0.5|0|10|17216
2|metabolic_process > 0.6 & cellular_process > 0.8|1|5|4837
3|metabolic_process <= 0.6 & cellular_process > 0.9|1|5|2359
4|cellular_anatomical_entity > 0.8 & 0.6 < cellular_process <= 0.8|1|6|2082
5|binding <= 0.5 & cellular_anatomical_entity > 0.1 & cellular_process <= 0.0 & biological_regulation <= 0.5|0|10|1403
6|binding <= 0.7 & cellular_anatomical_entity > 0.7 & cellular_process <= 0.5 & 0.5 < biological_regulation <= 0.9|0|10|841
7|0.5 < binding <= 0.5 & cellular_anatomical_entity > 0.8 & cellular_process <= 0.6 & biological_regulation <= 0.5|1|10|574
8|binding <= 0.7 & cellular_anatomical_entity > 0.7 & 0.5 < cellular_process <= 0.6 & 0.5 < biological_regulation <= 0.9|1|10|515
9|binding > 0.7 & cellular_anatomical_entity > 0.7 & cellular_process <= 0.6 & 0.5 < biological_regulation <= 0.9|1|8|472
10|0.5 < binding <= 0.7 & cellular_anatomical_entity > 0.9 & cellular_process <= 0.6 & biological_regulation <= 0.5|0|6|350
11|binding > 0.7 & cellular_anatomical_entity > 0.9 & cellular_process <= 0.6 & biological_regulation <= 0.5|1|6|268
12|binding <= 0.1 & cellular_anatomical_entity <= 0.1 & cellular_process <= 0.0 & biological_regulation <= 0.5|1|6|248
13|0.5 < binding <= 0.5 & 0.1 < cellular_anatomical_entity <= 0.8 & cellular_process <= 0.6 & biological_regulation <= 0.5|0|10|217
14|binding <= 0.5 & 0.1 < cellular_anatomical_entity <= 0.8 & 0.6 < cellular_process <= 0.8 & biological_regulation <= 0.7|0|7|210
15|0.1 < binding <= 0.5 & cellular_anatomical_entity <= 0.1 & cellular_process <= 0.0 & biological_regulation <= 0.5|0|6|157
16|cellular_anatomical_entity <= 0.8 & 0.6 < cellular_process <= 0.8 & biological_regulation > 0.7|1|6|101
17|0.5 < binding <= 0.5 & cellular_anatomical_entity <= 0.1 & cellular_process <= 0.1 & biological_regulation <= 0.5|1|7|93
18|cellular_anatomical_entity <= 0.7 & 0.0 < cellular_process <= 0.5 & biological_regulation > 0.7|0|5|86"""
def parse_rule(txt):
    d = {}
    for c in txt.split(" & "):
        m = re.fullmatch(r"([\d.]+) < (\w+) <= ([\d.]+)", c)
        if m: d[m[2]] = (float(m[1]), float(m[3])); continue
        m = re.fullmatch(r"(\w+) > ([\d.]+)", c)
        if m: d[m[1]] = (float(m[2]), None); continue
        m = re.fullmatch(r"(\w+) <= ([\d.]+)", c); d[m[1]] = (None, float(m[2]))
    return frozenset(d.items())
paper = [dict(n=int(a), rule=b, key=(parse_rule(b), int(c)), models=int(d), pairs=int(e))
         for a, b, c, d, e in (l.split("|") for l in PAPER.splitlines())]
def fmt(key):
    conds, cls = key; parts = []
    for f, (lo, hi) in sorted(conds):
        parts.append(f"{lo} < {f} <= {hi}" if lo is not None and hi is not None else f"{f} > {lo}" if lo is not None else f"{f} <= {hi}")
    return " & ".join(parts) + f"  => {'+' if cls else '-'}"
def rules_of(clf):
    t = clf.tree_; L, R = t.children_left.copy(), t.children_right.copy(); feat = t.feature.copy()
    dec = t.value.argmax(axis=2).flatten()
    leaf = lambda i: L[i] == -1 and R[i] == -1
    def prune(i):          # prune_index of the authors' ML.py, on copies of the arrays
        if not leaf(L[i]): prune(L[i])
        if not leaf(R[i]): prune(R[i])
        if leaf(L[i]) and leaf(R[i]) and dec[i] == dec[L[i]] == dec[R[i]]: L[i] = R[i] = -1; feat[i] = -2
    if not leaf(0): prune(0)
    out = {}
    def rec(i, path):
        if L[i] == -1:
            iv = {}
            for f, op, th in path:
                lo, hi = iv.get(f, (None, None))
                if op == ">": lo = th if lo is None else max(lo, th)
                else: hi = th if hi is None else min(hi, th)
                iv[f] = (lo, hi)
            r = lambda v: None if v is None else round(v, 1)
            out[i] = (frozenset((names[f], (r(lo), r(hi))) for f, (lo, hi) in iv.items()), int(dec[i])); return
        rec(L[i], path + [(feat[i], "<=", t.threshold[i])]); rec(R[i], path + [(feat[i], ">", t.threshold[i])])
    rec(0, [])
    def node_of(x):
        i = 0
        while L[i] != -1: i = L[i] if x[feat[i]] <= t.threshold[i] else R[i]
        return i
    return out, node_of
result = {}
for seed in SEEDS:
    freq = collections.Counter(); npairs = collections.Counter(); waf = []
    for te in folds:
        m = np.zeros(len(y), bool); m[te] = True
        clf = DecisionTreeClassifier(max_depth=6, random_state=seed).fit(X[~m], y[~m])
        rules, node_of = rules_of(clf)
        for key in set(rules.values()): freq[key] += 1
        for i in te: npairs[rules[node_of(X[i])]] += 1
    frequent = sorted([k for k, c in freq.items() if c >= 5], key=lambda k: -npairs[k])
    def close(a, b):       # same features, same class, every bound within 0.1
        if a[1] != b[1]: return False
        da, db = dict(a[0]), dict(b[0])
        if set(da) != set(db): return False
        for f in da:
            for u, v in zip(da[f], db[f]):
                if (u is None) != (v is None) or (u is not None and abs(u - v) > 0.1 + 1e-9): return False
        return True
    comp = []
    for p in paper:
        exact = freq.get(p["key"], 0)
        near = [k for k in freq if close(k, p["key"])]
        comp.append(dict(n=p["n"], rule=fmt(p["key"]), paper_models=p["models"], paper_pairs=p["pairs"],
                         ours_models_exact=exact, ours_pairs_exact=npairs.get(p["key"], 0),
                         ours_models_near=max([freq[k] for k in near], default=0)))
    result[seed] = dict(n_frequent=len(frequent), n_paper=len(paper),
                        n_paper_exact_found=sum(c["ours_models_exact"] > 0 for c in comp),
                        n_paper_exact_frequent=sum(c["ours_models_exact"] >= 5 for c in comp),
                        n_paper_near_frequent=sum(c["ours_models_near"] >= 5 for c in comp),
                        n_ours_frequent_in_paper=sum(any(close(k, p["key"]) for p in paper) for k in frequent),
                        comparison=comp, ours=[dict(rule=fmt(k), models=freq[k], pairs=npairs[k]) for k in frequent])
    r = result[seed]
    print(f"seed {seed}: our frequent rules {r['n_frequent']}; paper rules found exactly in >=1 model {r['n_paper_exact_found']}/18, "
          f"in >=5 models {r['n_paper_exact_frequent']}/18, near (bounds +-0.1) in >=5 models {r['n_paper_near_frequent']}/18; "
          f"our frequent rules matching a paper rule (near) {r['n_ours_frequent_in_paper']}/{r['n_frequent']}")
for c in result[SEEDS[0]]["comparison"]:
    print("%2d %-110s paper %2d/%5d  ours exact %2d/%5d near %2d" % (c["n"], c["rule"], c["paper_models"], c["paper_pairs"],
          c["ours_models_exact"], c["ours_pairs_exact"], c["ours_models_near"]))
print("our frequent rules (seed %d):" % SEEDS[0])
for o in result[SEEDS[0]]["ours"]: print("  %2d models %5d pairs  %s" % (o["models"], o["pairs"], o["rule"]))
json.dump(result, open("dt6_rules.json", "w"), indent=1)


In [ ]:
%%writefile plaus2.py
"""Quantitative Plausibility proxy: does an explanation's top aspect agree with external pathway knowledge?
usage: python3 plaus2.py GMT_NAME [CFGS e.g. seekx|rf,seekx|xgb] [TAG]     (reads {GMT_NAME}_msigdb751.gmt: REACTOME / KEGG from MSigDB 7.5.1, or REACTOME_LIVE
       written by get_reactome_live.py from the current reactome.org UniProt2Reactome file)
For every explained pair with y=1 and pred=1 (all rec_*_{rf,xgb}_f*.json), take the smallest gene set (10..300 members in the
GAF universe) that contains both proteins. Reference relevance of an aspect = over-representation (one-sided hypergeometric,
BH over the 51 aspects) of the OTHER members of that pathway (the two proteins excluded) among proteins annotated under the
aspect, versus all GAF proteins. Annotations used for the reference exclude evidence IEA and anything assigned_by Reactome,
so the reference does not reuse Reactome-derived GO annotations.
A pair is informative when it has >=2 players and at least one relevant and one non-relevant player.
Metrics over the pair's players: hit@1, precision@3, NDCG@3 (gain = -log10 p, capped at 10).
Baselines: random order (exact expectation for hit@1, 200 permutations otherwise) and, for KGsim2vec, ranking players by
their raw similarity feature (model-free)."""
import sys, glob, json, math, random, collections
import numpy as np
from scipy.stats import hypergeom, wilcoxon
sys.path.insert(0, ".")
from kgsim import KG

GMT = sys.argv[1]
CFGS = sys.argv[2].split(",") if len(sys.argv) > 2 else ["kg|rf", "kg|xgb", "seek|rf", "seek|xgb", "hyb|rf", "hyb|xgb"]
TAG = sys.argv[3] if len(sys.argv) > 3 else ""
K = np.load("feats_kg.npz"); keys = [tuple(k) for k in K["keys"]]; aspects = list(K["aspects"])
ci = aspects.index("GO_0032991"); keep = [j for j in range(52) if j != ci]; A = [aspects[j] for j in keep]
kg = KG("godata/go-basic.owl", "godata/goa_human.gaf", aspects); sub = [kg.sub[j] for j in keep]
Xpy = K["Xpy"][:, keep]

# filtered annotations (no IEA, not assigned by Reactome) and symbol -> accession
filt = collections.defaultdict(set); sym2acc = collections.defaultdict(set)
for l in open("godata/goa_human.gaf"):
    if l.startswith("!"): continue
    f = l.rstrip("\n").split("\t")
    if len(f) < 15: continue
    sym2acc[f[2]].add(f[1])
    if "NOT" in f[3] or f[6] == "IEA" or f[14] == "Reactome": continue
    filt[f[1]].add("GO_" + f[4].split(":")[1])
U = sorted(filt); NU = len(U)
asp_has = np.array([[bool(filt[p] & sub[a]) for a in range(len(A))] for p in U])  # NU x 51
uidx = {p: i for i, p in enumerate(U)}
bg = asp_has.sum(0)

sets = []
for l in open(f"{GMT}_msigdb751.gmt"):
    f = l.rstrip("\n").split("\t")
    mem = {a for s in f[2:] for a in (sym2acc.get(s) or {s})} & set(uidx)   # members may be gene symbols or UniProt accessions
    if 10 <= len(mem) <= 300: sets.append((f[0], mem))
member_of = collections.defaultdict(list)
for j, (_, mem) in enumerate(sets):
    for p in mem: member_of[p].append(j)
print(GMT, "gene sets kept", len(sets), "universe", NU, flush=True)

relcache = {}
def relevance(j, p1, p2):
    key = (j, p1, p2)
    if key in relcache: return relcache[key]
    M = [uidx[p] for p in sets[j][1] if p not in (p1, p2)]
    cnt = asp_has[M].sum(0); n = len(M)
    pv = np.array([hypergeom.sf(cnt[a] - 1, NU, bg[a], n) if bg[a] > 0 else 1.0 for a in range(len(A))])
    order = np.argsort(pv); q = np.empty_like(pv); m = len(pv); prev = 1.0
    for rank, a in reversed(list(enumerate(order))):
        prev = min(prev, pv[a] * m / (rank + 1)); q[a] = prev
    relcache[key] = (q < 0.05, np.minimum(-np.log10(np.maximum(pv, 1e-300)), 10.0))
    return relcache[key]

def players(p1, p2):
    t1, t2 = kg.ann[p1], kg.ann[p2]
    return [i for i in range(len(A)) if t1 & sub[i] and t2 & sub[i]]
def ndcg3(order, gain):
    dcg = sum(gain[a] / math.log2(r + 2) for r, a in enumerate(order[:3]))
    ideal = sorted((gain[a] for a in order), reverse=True)
    idcg = sum(g / math.log2(r + 2) for r, g in enumerate(ideal[:3]))
    return dcg / idcg if idcg > 0 else 0.0

out = {}; cases = []
for cfg in CFGS:
    pipe, model = cfg.split("|"); recs = []
    for fn in sorted(glob.glob(f"rec_{pipe}_{model}_f*.json")): recs += json.load(open(fn))["records"]
    methods = [k[4:] for k in recs[0] if k.startswith("phi_")]
    res = collections.defaultdict(lambda: collections.defaultdict(list)); n_tp = n_shared = 0
    for r in recs:
        if not (r["y"] == 1 and r["pred"] == 1): continue
        n_tp += 1
        p1, p2 = keys[r["i"]]
        shared = set(member_of.get(p1, ())) & set(member_of.get(p2, ()))
        if not shared: continue
        n_shared += 1
        j = min(shared, key=lambda j: len(sets[j][1]))
        rel, gain = relevance(j, p1, p2)
        # players: KGsim2vec players for kg; for seek/hyb, aspects with non-zero KAS or LOO attribution (= SEEK players)
        pl = players(p1, p2) if pipe.startswith("kg") else sorted({A.index(a) for m in ("D", "LOO") for a in r[f"phi_{m}"]})
        nrel = int(sum(rel[i] for i in pl))
        if len(pl) < 2 or nrel == 0 or nrel == len(pl): continue
        rr = random.Random(r["i"])
        orders = {}
        for m in methods:
            phi = r[f"phi_{m}"]; tb = {i: rr.random() for i in pl}
            orders[m] = sorted(pl, key=lambda i: (-phi.get(A[i], 0.0), tb[i]))
        if pipe.startswith("kg"):
            tb = {i: rr.random() for i in pl}
            orders["Sim"] = sorted(pl, key=lambda i: (-Xpy[r["i"], i], tb[i]))
        for m, o in orders.items():
            res[m]["hit1"].append(float(rel[o[0]])); res[m]["p3"].append(float(np.mean([rel[a] for a in o[:3]])))
            res[m]["ndcg3"].append(ndcg3(o, gain))
        res["random"]["hit1"].append(nrel / len(pl))
        perms = [rr.sample(pl, len(pl)) for _ in range(200)]
        res["random"]["p3"].append(float(np.mean([np.mean([rel[a] for a in o[:3]]) for o in perms])))
        res["random"]["ndcg3"].append(float(np.mean([ndcg3(o, gain) for o in perms])))
        if cfg == "kg|rf":
            cases.append(dict(pair=r["pair"], pathway=sets[j][0], size=len(sets[j][1]), players=[kg.label.get(A[i], A[i]) for i in pl],
                              relevant=[kg.label.get(A[i], A[i]) for i in pl if rel[i]],
                              top3={m: [kg.label.get(A[i], A[i]) for i in o[:3]] for m, o in orders.items()}))
    summ = dict(n_tp=n_tp, n_shared=n_shared, n_informative=len(res["random"]["hit1"]), metrics={}, wilcoxon_vs_D={})
    for m, d in res.items():
        summ["metrics"][m] = {k: dict(mean=float(np.mean(v)),
                                      ci95=[float(x) for x in np.percentile([np.mean(np.random.default_rng(b).choice(v, len(v))) for b in range(1000)], [2.5, 97.5])])
                              for k, v in d.items()}
    for m in res:
        if m == "D": continue
        for k in ("hit1", "ndcg3"):
            a, b = np.array(res["D"][k]), np.array(res[m][k])
            summ["wilcoxon_vs_D"][f"{m}|{k}"] = float(wilcoxon(a, b).pvalue) if np.any(a != b) else 1.0
    out[cfg] = summ
    print(cfg, "TP", n_tp, "shared", n_shared, "informative", summ["n_informative"])
    for m in summ["metrics"]:
        mm = summ["metrics"][m]
        print("   %-16s hit@1 %.3f  p@3 %.3f  ndcg@3 %.3f" % (m, mm["hit1"]["mean"], mm["p3"]["mean"], mm["ndcg3"]["mean"]),
              "" if m == "D" else "p(hit1)=%.2g p(ndcg)=%.2g" % (summ["wilcoxon_vs_D"].get(f"{m}|hit1", 1), summ["wilcoxon_vs_D"].get(f"{m}|ndcg3", 1)))
json.dump(dict(gmt=GMT, results=out, cases=cases), open(f"plaus_{GMT}{TAG}.json", "w"))


In [ ]:
%%writefile analyze_e1.py
"""E1 summary: KAS (D) vs SEEK's own explainer mapped to aspects (SEEKN graded-necessary, SEEKS graded-sufficient), LOO,
random, on the SEEK view; deletion/insertion under both operators ('ks' = delete annotations + recompute, KAS's value
function; 'dca' = drop DCA vectors, SEEK's operator). Same aggregation as analyze_v2.py (deletion = mean of k=1..5,
insertion = mean of k=1..5; empty explanation -> deletion stays at the original confidence, insertion at the all-removed
state). usage: python3 analyze_e1.py  -> e1_summary.json, prints a table"""
import json, glob, collections
import numpy as np
from scipy.stats import wilcoxon
M = ["D", "LOO", "SEEKN", "SEEKS", "random"]; out = {}
for model in ("rf", "xgb"):
    conf = {}
    for fn in glob.glob(f"rec_seek_{model}_f*_*.json"):
        for r in json.load(open(fn))["records"]: conf[r["i"]] = r["conf"]
    R = []
    for fn in sorted(glob.glob(f"rec_seekx_{model}_f*.json")): R += json.load(open(fn))["records"]
    if not R: continue
    s = dict(n=len(R), folds=len(glob.glob(f"rec_seekx_{model}_f*.json")))
    for sp in ("ks", "dca"):
        dels, inss = {}, {}
        for m in M:
            dels[m] = np.array([np.mean(r[f"del_{sp}_{m}"]) if r.get(f"del_{sp}_{m}") else conf[r["i"]] for r in R])
            inss[m] = np.array([np.mean(r[f"ins_{sp}_{m}"][1:]) if r.get(f"ins_{sp}_{m}") else
                                (r[f"ins_{sp}_random"][0] if r.get(f"ins_{sp}_random") else conf[r["i"]]) for r in R])
            s[f"del_{sp}_{m}"] = float(dels[m].mean()); s[f"ins_{sp}_{m}"] = float(inss[m].mean())
        for m in M[1:]:
            for nm, arr, alt in (("del", dels, "less"), ("ins", inss, "greater")):
                a, b = arr["D"], arr[m]
                s[f"p_{nm}_{sp}_D_vs_{m}"] = float(wilcoxon(a, b, alternative=alt).pvalue) if np.any(a != b) else 1.0
                s[f"Dbetter_{nm}_{sp}_{m}"] = float(np.mean((a < b) if nm == "del" else (a > b)))
    for m in M[:-1]:
        s[f"pars_{m}"] = float(np.mean([r[f"size_{m}"] for r in R])); s[f"cov_{m}"] = float(np.mean([r[f"cov_{m}"] for r in R]))
    for m in ("LOO", "SEEKN", "SEEKS"):
        s[f"top1same_D_{m}"] = float(np.mean([r[f"top1same_D_{m}"] for r in R])); s[f"jac3_D_{m}"] = float(np.mean([r[f"jac3_D_{m}"] for r in R]))
    s["dca_mean"] = float(np.mean([r["n_dca"] for r in R])); s["dca_median"] = float(np.median([r["n_dca"] for r in R]))
    s["dca_outside_aspects_frac"] = float(sum(r["n_dca_noasp"] for r in R) / max(1, sum(r["n_dca"] for r in R)))
    s["pairs_with_necessary"] = float(np.mean([r["n_nec"] > 0 for r in R])); s["necessary_mean"] = float(np.mean([r["n_nec"] for r in R]))
    s["sufficient_mean"] = float(np.mean([r["n_suf"] for r in R]))
    s["time_SEEK_median"] = float(np.median([r["time_SEEK"] for r in R]))
    A = json.load(open(sorted(glob.glob(f"rec_seekx_{model}_f*.json"))[0]))["aspects"]
    hit = [int(r["top1_D"] is not None and A.index(r["top1_D"]) in r["nec_asp"]) for r in R if r["nec_asp"]]
    s["KAS_top1_in_SEEK_necessary_aspects"] = float(np.mean(hit)) if hit else None
    out[model] = s
    print(f"== SEEK view, {model.upper()}: {s['n']} pairs ({s['folds']} folds) ==")
    print("  %-8s %8s %8s %8s %8s %6s %6s" % ("method", "del_ks", "ins_ks", "del_dca", "ins_dca", "pars", "cov"))
    for m in M:
        print("  %-8s %8.3f %8.3f %8.3f %8.3f %6s %6s" % (m, s[f"del_ks_{m}"], s[f"ins_ks_{m}"], s[f"del_dca_{m}"], s[f"ins_dca_{m}"],
              "%.2f" % s[f"pars_{m}"] if m != "random" else "", "%.3f" % s[f"cov_{m}"] if m != "random" else ""))
    for m in M[1:]:
        print("  D vs %-6s p(del_ks) %.2g p(ins_ks) %.2g p(del_dca) %.2g p(ins_dca) %.2g | D better del_ks %.2f del_dca %.2f" % (
            m, s[f"p_del_ks_D_vs_{m}"], s[f"p_ins_ks_D_vs_{m}"], s[f"p_del_dca_D_vs_{m}"], s[f"p_ins_dca_D_vs_{m}"],
            s[f"Dbetter_del_ks_{m}"], s[f"Dbetter_del_dca_{m}"]))
    print("  agreement with KAS: top-1 same LOO %.2f SEEKN %.2f SEEKS %.2f | Jaccard@3 LOO %.2f SEEKN %.2f SEEKS %.2f" % tuple(
        [s[f"top1same_D_{m}"] for m in ("LOO", "SEEKN", "SEEKS")] + [s[f"jac3_D_{m}"] for m in ("LOO", "SEEKN", "SEEKS")]))
    print("  SEEK native: DCAs/pair mean %.1f (median %.0f), outside every aspect %.1f%%; pairs with >=1 necessary DCA %.1f%%; "
          "necessary %.2f, sufficient %.2f per pair; KAS top-1 inside a SEEK-necessary aspect %.2f; time %.3fs" % (
          s["dca_mean"], s["dca_median"], 100 * s["dca_outside_aspects_frac"], 100 * s["pairs_with_necessary"], s["necessary_mean"],
          s["sufficient_mean"], s["KAS_top1_in_SEEK_necessary_aspects"] or float("nan"), s["time_SEEK_median"]))
json.dump(out, open("e1_summary.json", "w"), indent=1)


In [ ]:
%%writefile analyze_sym.py
"""Summary of roar_sym_{PIPE}_{MODEL}.json: WAF drop (base median - variant median) for MoRF and LeRF, k = 1..3,
gap = WAF(LeRF) - WAF(MoRF) (median over folds), mask-only accuracy, one-sided Wilcoxon of KAS (D) vs each method.
usage: python3 analyze_sym.py PIPE MODEL"""
import sys, json
import numpy as np
PIPE, MODEL = sys.argv[1], sys.argv[2]
o = json.load(open(f"roar_sym_{PIPE}_{MODEL}.json")); base = np.median(o["base"])
methods = sorted({k.split("|")[0] for k in o["res"]} - {"random"}, key=lambda m: (m != "D", m)) + ["random"]
print(f"== two-direction ROAR, {PIPE} + {MODEL.upper()}, {o['n']} pairs, base WAF {base:.4f} ==")
print("  %-9s | %-23s | %-23s | %-23s | %s" % ("method", "MoRF drop k=1/2/3 (up)", "LeRF drop k=1/2/3 (down)", "gap k=1/2/3 (up)", "mask-only MoRF/LeRF k=1"))
for m in methods:
    mo = [base - np.median(o["res"][f"{m}|MoRF|{k}"]) for k in (1, 2, 3) if f"{m}|MoRF|{k}" in o["res"]]
    le = [base - np.median(o["res"][f"{m}|LeRF|{k}"]) for k in (1, 2, 3) if f"{m}|LeRF|{k}" in o["res"]]
    gp = [np.median(np.array(o["res"][f"{m}|LeRF|{k}"]) - np.array(o["res"][f"{m}|MoRF|{k}"])) for k in (1, 2, 3) if f"{m}|LeRF|{k}" in o["res"]]
    mk = "%.3f / %s" % (o["mask"].get(f"{m}|MoRF|1", float("nan")), "%.3f" % o["mask"][f"{m}|LeRF|1"] if f"{m}|LeRF|1" in o["mask"] else "-")
    f = lambda v: " ".join("%.3f" % x for x in v) if v else "-"
    print("  %-9s | %-23s | %-23s | %-23s | %s" % (m, f(mo), f(le), f(gp), mk))
for k, v in sorted(o.get("p", {}).items()):
    if v < 0.05 or k.endswith("_1"): print("  p", k, "%.3g" % v)


## 1. Đặc trưng (bỏ qua nếu đã có)

In [ ]:
import os
if not os.path.exists('feats_kg.npz'):
    !python build_feats.py && cp feats.npz feats_kg.npz
if not os.path.exists('feats_seek.npz'):
    !python build_seek_feats.py
print('feats_kg.npz', os.path.exists('feats_kg.npz'), '| feats_seek.npz', os.path.exists('feats_seek.npz'))

In [ ]:
import os, subprocess, time
from concurrent.futures import ThreadPoolExecutor
def run_jobs(jobs, workers=None):
    """jobs = [(output_file, [command...], log_file)]; a job whose output already exists is skipped."""
    t0 = time.time()
    def one(j):
        out, cmd, log = j
        if os.path.exists(out): return out + ': đã có, bỏ qua'
        t = time.time(); r = subprocess.run(cmd, capture_output=True, text=True); open(log, 'w').write(r.stdout + r.stderr)
        return out + (f': xong sau {(time.time()-t)/60:.1f} phút' if r.returncode == 0 else ': LỖI\n' + r.stderr[-1500:])
    with ThreadPoolExecutor(max_workers=workers or os.cpu_count()) as ex:
        for i, msg in enumerate(ex.map(one, jobs)): print(f'[{i+1}/{len(jobs)}, {(time.time()-t0)/60:.0f} phút]', msg, flush=True)

## E5. Bảng 9 của KGsim2vec: luật DT6 hay gặp
DT6 trên đặc trưng SML của 51 aspect, 10 partition; làm tròn ngưỡng 1 chữ số; đếm luật có ở ít nhất 5/10 mô hình; so với Bảng 9 của bài báo. Chạy 5 seed để kiểm tra độ ổn định.

In [ ]:
!python dt6_rules.py 0,1,2,3,4 | tee e5_dt6_rules.txt

## E4. Shapley chính xác so với ước lượng hoán vị
300 cặp có 6 ≤ k ≤ 8 (fold 1, RF) trên view KGsim2vec và SEEK; M = 25, 50, 100, 200 hoán vị, 3 seed.

In [ ]:
run_jobs([('perm_kg_rf_f1.json', ['python', 'perm_check.py', 'kg', 'rf', '1', '300'], 'log_e4_kg.txt'),
          ('perm_seek_rf_f1.json', ['python', 'perm_check.py', 'seek', 'rf', '1', '300'], 'log_e4_seek.txt')])
!tail -4 log_e4_kg.txt; tail -4 log_e4_seek.txt

## E1. KAS so với bộ giải thích gốc của SEEK
Cần `rec_seek_{rf,xgb}_f*.json` (2,200 cặp mỗi mô hình, từ lần chạy chính). Nếu chưa có, ô đầu tiên chạy lại `explain_v2.py` cho view SEEK (khoảng 1–2 giờ).
- SEEKN: necessary có thang (xác suất giảm khi bỏ một DCA), cộng theo aspect chứa DCA.
- SEEKS: sufficient có thang (xác suất khi chỉ giữ một DCA), lấy DCA ủng hộ dự đoán nhất trong aspect.
- Hai toán tử xoá: `ks` (xoá chú giải rồi tính lại, của KAS) và `dca` (bỏ vector DCA, của SEEK).

In [ ]:
run_jobs([(f'rec_seek_{m}_f{f}_{400 if f == 1 else 200}.json', ['python', 'explain_v2.py', 'seek', m, str(f), str(400 if f == 1 else 200)],
           f'log_rec_seek_{m}_{f}.txt') for m in ['rf', 'xgb'] for f in range(1, 11)])
run_jobs([(f'rec_seekx_{m}_f{f}.json', ['python', 'seek_native.py', m, str(f)], f'log_e1_{m}_{f}.txt') for m in ['rf', 'xgb'] for f in range(1, 11)])

In [ ]:
import os
if not os.path.exists('REACTOME_msigdb751.gmt'):
    !python get_msigdb.py
!python analyze_e1.py | tee e1_summary.txt
!python plaus2.py REACTOME 'seekx|rf,seekx|xgb' _seekx | tee e1_plaus_reactome.txt
!python plaus2.py KEGG 'seekx|rf,seekx|xgb' _seekx | tee e1_plaus_kegg.txt

## E2. ROAR đầy đủ 47,142 cặp cho view SEEK (KAS, LOO, SEEKN, SEEKS, ngẫu nhiên)
Bước 1 giải thích mọi cặp (mỗi cặp do mô hình của fold mà nó là cặp test). Bước 2 bỏ top-k (MoRF) hoặc bottom-k (LeRF) aspect, huấn luyện lại RF trên 10 partition gốc, kèm kiểm tra rò rỉ mask.

In [ ]:
NCPU = os.cpu_count(); NPARTS = max(1, NCPU // 2)
run_jobs([(f'all_seek_rf_f{f}_p{p}.json' if NPARTS > 1 else f'all_seek_rf_f{f}.json',
           ['python', 'explain_all_seek.py', 'rf', str(f), str(p), str(NPARTS)], f'log_e2_all_{f}_{p}.txt') for f in range(1, 11) for p in range(NPARTS)])

In [ ]:
if not os.path.exists('roar_sym_seek_rf.json') or 'gap' not in open('roar_sym_seek_rf.json').read():
    !NJOBS={os.cpu_count()} python roar_sym.py seek rf | tee log_e2_roar_sym_seek.txt
!python analyze_sym.py seek rf | tee e2_roar_sym_seek.txt

## E3. ROAR hai chiều cho view KGsim2vec (KAS, LOO, FS0, TreeSHAP, LIME, ngẫu nhiên)
Dùng lại `all_kg_rf_f*.json` của ROAR đầy đủ ngày 06/10 nếu có; nếu chưa có, ô đầu chạy lại (khoảng 2–3 giờ).

In [ ]:
import glob
if len(glob.glob('all_kg_rf_f*.json')) < 10:
    NPARTS = max(1, os.cpu_count() // 2)
    run_jobs([(f'all_kg_rf_f{f}_p{p}.json' if NPARTS > 1 else f'all_kg_rf_f{f}.json', ['python', 'explain_all.py', 'rf', str(f), str(p), str(NPARTS)],
               f'log_all_kg_{f}_{p}.txt') for f in range(1, 11) for p in range(NPARTS)])
if not os.path.exists('roar_sym_kg_rf.json') or 'gap' not in open('roar_sym_kg_rf.json').read():
    !NJOBS={os.cpu_count()} python roar_sym.py kg rf | tee log_e3_roar_sym_kg.txt
!python analyze_sym.py kg rf | tee e3_roar_sym_kg.txt

## Tổng hợp
Các file tóm tắt: `e5_dt6_rules.txt`, `perm_*.json`, `e1_summary.txt`, `e1_plaus_*.txt`, `e2_roar_sym_seek.txt`, `e3_roar_sym_kg.txt`. Gửi các file này (hoặc cả thư mục) cho Claude để đối chiếu với kết quả chạy trong sandbox.

In [ ]:
!ls -la *.txt *.json | grep -E 'e[1-5]_|perm_|roar_sym|dt6' ; for f in e5_dt6_rules.txt e1_summary.txt e2_roar_sym_seek.txt e3_roar_sym_kg.txt; do echo "=== $f"; head -30 $f; done